In [20]:
import pandas as pd
import numpy as np

In [21]:
cc_calls = pd.read_csv(
    "../../data/01_raw/raw_cc_calls.csv",
    low_memory=False
)

print(f"Shape: {cc_calls.shape}")
cc_calls.shape

Shape: (32882, 33)


(32882, 33)

In [22]:
cc_calls.info()
cc_calls.head()

<class 'pandas.DataFrame'>
RangeIndex: 32882 entries, 0 to 32881
Data columns (total 33 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Contact_ID                                32882 non-null  float64
 1   Call_Date                                 32882 non-null  str    
 2   Direction                                 32882 non-null  str    
 3   cc_care_package                           32744 non-null  str    
 4   cc_care_package_discussed                 32744 non-null  str    
 5   cc_urgency_getting_on_site                32744 non-null  str    
 6   cc_external_consultant                    32744 non-null  str    
 7   cc_agent_cross_sell_attempt               32744 non-null  str    
 8   cc_customer_issues_concerns               32744 non-null  str    
 9   cc_business_struggles_financial_hardship  32744 non-null  str    
 10  cc_call_initiated_by                      327

,Contact_ID,Call_Date,Direction,cc_care_package,cc_care_package_discussed,cc_urgency_getting_on_site,cc_external_consultant,cc_agent_cross_sell_attempt,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,...,cc_contractor_sentiment_overall_score,cc_contractor_sentiment_issues_score,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,Co_Ref,Analysed_Call,Call_Year
0,6.255130e+11,08-05-2025,OUT_BOUND,Standard,Yes,No,No,No,Yes,Yes,...,30,20,Yes,Yes,No,Yes,Yes,HV3323,1,2025
1,5.910870e+11,25-11-2024,OUT_BOUND,Standard,Yes,No,No,No,Yes,No,...,0,0,Yes,Yes,No,Yes,Yes,PJ7066,1,2024
2,5.650910e+11,23-10-2024,IN_BOUND,Standard,Yes,No,No,No,Yes,No,...,40,20,Yes,Yes,No,Yes,Yes,DP6030,1,2024
3,5.939750e+11,13-01-2025,IN_BOUND,Premier,Yes,No,No,No,Yes,Yes,...,40,30,Yes,Yes,Yes,Yes,Yes,AM2413,1,2025
4,6.222820e+11,19-03-2025,IN_BOUND,Standard,Yes,No,No,No,Yes,Yes,...,40,20,Yes,Yes,No,Yes,Yes,ED6707,1,2025


In [23]:
cc_calls = cc_calls.drop_duplicates()
print("Shape after duplicate removal:", cc_calls.shape)

Shape after duplicate removal: (32789, 33)


In [24]:
cc_calls.columns = (
    cc_calls.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [25]:
for col in cc_calls.columns:
    if "date" in col:
        cc_calls[col] = pd.to_datetime(
            cc_calls[col], errors="coerce"
        )

In [26]:
missing_df = (
    cc_calls.isnull()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_df.head(20)

call_date                                   58.537924
co_ref                                       3.516423
cc_issues_within_questionnaire               1.421208
cc_call_initiated_by                         0.417823
cc_care_package_discussed                    0.417823
cc_agent_cross_sell_attempt                  0.417823
cc_external_consultant                       0.417823
cc_customer_issues_concerns                  0.417823
cc_care_package                              0.417823
cc_urgency_getting_on_site                   0.417823
cc_business_struggles_financial_hardship     0.417823
cc_contractor_suggest_leave                  0.289731
cc_refund_discussed                          0.289731
cc_contractor_sentiment_overall_score        0.289731
cc_contractor_sentiment_end_score            0.289731
cc_contractor_sentiment                      0.289731
cc_contractor_sentiment_issues_score         0.289731
cc_pricing_mentioned                         0.289731
cc_contractor_complained    

In [27]:
cat_cols = cc_calls.select_dtypes(include="object").columns.tolist()
num_cols = cc_calls.select_dtypes(include=np.number).columns.tolist()

# protect id columns
id_cols = [col for col in num_cols if "id" in col or "ref" in col]
num_cols = [col for col in num_cols if col not in id_cols]

print("Categorical:", len(cat_cols))
print("Numeric:", len(num_cols))

Categorical: 29
Numeric: 2


C:\Users\NuluShreya\AppData\Local\Temp\ipykernel_43348\352053162.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = cc_calls.select_dtypes(include="object").columns.tolist()


In [28]:
# -------------------------------
# categorical columns
# -------------------------------
for col in cat_cols:
    if cc_calls[col].isnull().sum() > 0:
        cc_calls[col] = cc_calls[col].fillna("No_CC_Interaction")

# -------------------------------
# numeric columns
# -------------------------------
for col in num_cols:
    if cc_calls[col].isnull().sum() > 0:
        unique_vals = cc_calls[col].dropna().nunique()

        # binary columns
        if unique_vals <= 2:
            cc_calls[col] = cc_calls[col].fillna(0)

        # count style
        elif any(word in col.lower() for word in ["count", "num", "calls", "attempts"]):
            cc_calls[col] = cc_calls[col].fillna(0)

        # continuous metrics
        else:
            cc_calls[col] = cc_calls[col].fillna(
                cc_calls[col].median()
            )

In [29]:
cc_calls.dtypes

contact_id                                         float64
call_date                                   datetime64[us]
direction                                              str
cc_care_package                                        str
cc_care_package_discussed                              str
cc_urgency_getting_on_site                             str
cc_external_consultant                                 str
cc_agent_cross_sell_attempt                            str
cc_customer_issues_concerns                            str
cc_business_struggles_financial_hardship               str
cc_call_initiated_by                                   str
cc_questionnaire_completion                            str
cc_chasing_response                                    str
cc_issues_within_questionnaire                         str
cc_login_issues                                        str
cc_platform_issues                                     str
cc_dissatisfaction_time_to_complete                    s

In [30]:
cc_calls.to_csv(
    "../../data/02_processed/processed_cc_calls.csv",
    index=False
)

In [31]:
# # 🔍 COMPREHENSIVE CC_CALLS DATA QUALITY ANALYSIS

# print("="*70)
# print("CC_CALLS DATA: QUALITY & COLUMN ANALYSIS")
# print("="*70)

# print(f"\nDataset Shape: {cc_calls.shape}")
# print(f"Total nulls: {cc_calls.isnull().sum().sum()}\n")

# # 1. Check ID/Tracking columns
# print("1️⃣  ID/TRACKING COLUMNS (cardinality check):")
# id_keywords = ["id", "ref", "call_id", "unnamed", "row"]
# for col in cc_calls.columns:
#     if any(keyword in col.lower() for keyword in id_keywords):
#         unique_pct = cc_calls[col].nunique() / len(cc_calls) * 100
#         print(f"   {col}: {cc_calls[col].nunique()} unique ({unique_pct:.1f}%)")

# # 2. Check for completely empty/sparse columns
# print("\n2️⃣  EMPTY/SPARSE COLUMNS (>80% missing):")
# sparse_found = False
# for col in cc_calls.columns:
#     missing_pct = cc_calls[col].isnull().sum() / len(cc_calls) * 100
#     if missing_pct > 80:
#         print(f"   {col}: {missing_pct:.1f}% missing")
#         sparse_found = True
# if not sparse_found:
#     print("   ✅ None found")

# # 3. Check low-variance columns (>90% one value)
# print("\n3️⃣  LOW-VARIANCE COLUMNS (>90% single value):")
# low_var_found = False
# for col in cc_calls.select_dtypes(include="object").columns:
#     if cc_calls[col].nunique() > 0:
#         top_pct = cc_calls[col].value_counts().head(1).values[0] / len(cc_calls) * 100
#         if top_pct > 90:
#             top_val = cc_calls[col].value_counts().head(1).index[0]
#             print(f"   {col}: '{top_val}' = {top_pct:.1f}%")
#             low_var_found = True
# if not low_var_found:
#     print("   ✅ None found")

# # 4. Check "Unknown" concentration
# print("\n4️⃣  'UNKNOWN' VALUE CONCENTRATION:")
# unknown_cols = []
# for col in cc_calls.select_dtypes(include="object").columns:
#     unknown_pct = (cc_calls[col] == "Unknown").sum() / len(cc_calls) * 100
#     if unknown_pct > 30:
#         print(f"   {col}: {unknown_pct:.1f}% Unknown")
#         unknown_cols.append(col)

# if not unknown_cols:
#     print("   ✅ No excessive Unknown values found")

# # 5. Check numeric columns
# print("\n5️⃣  NUMERIC COLUMNS - Value Distribution:")
# for col in cc_calls.select_dtypes(include=[np.number]).columns:
#     zero_count = (cc_calls[col] == 0).sum()
#     zero_pct = zero_count / len(cc_calls) * 100
#     print(f"   {col}: min={cc_calls[col].min():.2f}, max={cc_calls[col].max():.2f}, zeros={zero_pct:.1f}%")

# # 6. Column type summary
# print("\n6️⃣  COLUMN TYPE DISTRIBUTION:")
# print(f"   Categorical (object): {cc_calls.select_dtypes(include='object').shape[1]}")
# print(f"   Numeric (int/float): {cc_calls.select_dtypes(include=[np.number]).shape[1]}")
# print(f"   Datetime: {cc_calls.select_dtypes(include=['datetime64']).shape[1]}")

# print("\n" + "="*70)

In [32]:

# # 🧹 ADDITIONAL CC_CALLS CLEANING

# print("\n📋 PERFORMING ADDITIONAL CLEANING...\n")

# # 1. Drop ID columns with very high cardinality
# cols_to_drop = []
# for col in cc_calls.columns:
#     if col.lower() in ["co_ref", "customer_ref", "call_id"]:
#         unique_pct = cc_calls[col].nunique() / len(cc_calls) * 100
#         if unique_pct > 30:
#             cols_to_drop.append(col)
#             print(f"   ✅ Dropping '{col}' (high cardinality: {unique_pct:.1f}%)")

# if cols_to_drop:
#     cc_calls = cc_calls.drop(columns=cols_to_drop, errors='ignore')

# # 2. Drop extremely low-variance columns (>95% Unknown)
# low_var_cols = []
# for col in cc_calls.select_dtypes(include="object").columns:
#     if cc_calls[col].nunique() > 0:
#         top_pct = cc_calls[col].value_counts().head(1).values[0] / len(cc_calls) * 100
#         if top_pct > 95 and cc_calls[col].value_counts().head(1).index[0] == "Unknown":
#             low_var_cols.append(col)
#             print(f"   ✅ Dropping '{col}' ({top_pct:.1f}% Unknown)")

# if low_var_cols:
#     cc_calls = cc_calls.drop(columns=low_var_cols)

# # 3. Convert binary Yes/No columns to numeric
# print(f"\n   Converting binary columns to numeric...")
# binary_count = 0
# for col in cc_calls.select_dtypes(include="object").columns:
#     unique_vals = set(cc_calls[col].dropna().unique())
#     if unique_vals.issubset({'Yes', 'No'}) and len(unique_vals) > 1:
#         cc_calls[col] = cc_calls[col].map({'Yes': 1, 'No': 0}).astype('int64')
#         binary_count += 1

# if binary_count > 0:
#     print(f"   ✅ Converted {binary_count} binary columns")

# # 4. Final null check
# null_check = cc_calls.isnull().sum().sum()
# print(f"\n   Final nulls: {null_check}")

# # 5. Summary
# print(f"\n✅ AFTER ADDITIONAL CLEANING:")
# print(f"   Shape: {cc_calls.shape}")
# print(f"   Columns removed: {len(cols_to_drop) + len(low_var_cols)}")
# print(f"   Binary columns converted: {binary_count}")

# # 6. Save cleaned data
# cc_calls.to_csv(
#     "../../data/02_processed/processed_cc_calls.csv",
#     index=False
# )
# print(f"\n   ✅ Saved to processed_cc_calls.csv")

In [33]:

# # 📊 FINAL VERIFICATION

# print("\n📊 FINAL CC_CALLS DATA STATUS:\n")

# # Reload the saved data
# final_cc_calls = pd.read_csv("../../data/02_processed/processed_cc_calls.csv")

# print(f"Shape: {final_cc_calls.shape}")
# print(f"Nulls: {final_cc_calls.isnull().sum().sum()}")
# print(f"Memory: {final_cc_calls.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# print(f"\nColumn types:")
# col_types = final_cc_calls.dtypes.value_counts()
# for dtype, count in col_types.items():
#     print(f"   {dtype}: {count}")

# print(f"\n✅ CC_CALLS data is clean and ready for modeling!")

In [34]:

# # 🔄 FINAL NULL HANDLING

# print("\n🔄 HANDLING REMAINING NULLS...\n")

# cc_calls_processed = pd.read_csv("../../data/02_processed/processed_cc_calls.csv")

# print(f"Before final null handling: {cc_calls_processed.isnull().sum().sum()} nulls")

# # Handle datetime columns with NaT
# for col in cc_calls_processed.columns:
#     if cc_calls_processed[col].dtype == 'object':
#         # Try to detect if this looks like a date column
#         sample = cc_calls_processed[col].dropna().head(5)
#         if len(sample) > 0:
#             try:
#                 # If it can be converted to datetime, fill NaT values
#                 test_convert = pd.to_datetime(sample, errors='coerce')
#                 if test_convert.notna().all():
#                     cc_calls_processed[col] = pd.to_datetime(cc_calls_processed[col], errors='coerce')
#                     nat_count = cc_calls_processed[col].isna().sum()
#                     if nat_count > 0:
#                         # Fill with a default date
#                         cc_calls_processed[col] = cc_calls_processed[col].fillna(cc_calls_processed[col].median())
#                         print(f"   ✅ Filled {nat_count} NaT values in '{col}'")
#             except:
#                 pass

# # Fill remaining numeric nulls
# for col in cc_calls_processed.select_dtypes(include=[np.number]).columns:
#     if cc_calls_processed[col].isnull().sum() > 0:
#         cc_calls_processed[col] = cc_calls_processed[col].fillna(cc_calls_processed[col].median())
#         print(f"   ✅ Filled numeric nulls in '{col}'")

# print(f"\nAfter final null handling: {cc_calls_processed.isnull().sum().sum()} nulls")

# # Save final cleaned data
# cc_calls_processed.to_csv(
#     "../../data/02_processed/processed_cc_calls.csv",
#     index=False
# )

# print(f"\n✅ FINAL CC_CALLS STATUS:")
# print(f"   Shape: {cc_calls_processed.shape}")
# print(f"   Nulls: {cc_calls_processed.isnull().sum().sum()}")
# print(f"   Memory: {cc_calls_processed.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
# print(f"\n   ✅ Saved to processed_cc_calls.csv")